# Group 5 – PayShield Fintech
## Member 1: Data Engineering & Preprocessing

This notebook contains **only the Data Engineer tasks** for the Credit Card Fraud Detection mini-project.

### Responsibilities covered
1. Load and document the raw dataset
2. Validate the dataset schema
3. Inspect data types
4. Check and handle missing values
5. Check and handle duplicate records
6. Validate transaction IDs
7. Validate invalid values
8. Clean categorical/text fields
9. Convert date fields
10. Engineer reusable features
11. Handle transaction amount appropriately
12. Separate features and target
13. Perform a stratified train/test split
14. Build leakage-safe preprocessing
15. Scale numerical variables using training data only
16. Save processed train/test datasets
17. Generate a data-quality audit log
18. Provide a reproducible preprocessing function for `src/preprocessing.py`

> **Important:** EDA interpretation, model building, model evaluation, dashboard work, and business recommendations belong to other project tasks and are intentionally not performed here.


In [14]:
# ============================================
# 1. IMPORT LIBRARIES AND CONFIGURATION
# ============================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Project paths

The project is designed to run from VS Code/GitHub without using a personal Windows path.

Expected structure:

```text
project/
│
├── data/
│   ├── raw/
│   │   └── credit_card_fraud.csv
│   └── processed/
│
├── notebooks/
│   └── Member1_Data_Engineer.ipynb
│
├── src/
│   └── preprocessing.py
│
├── requirements.txt
└── README.md
```


In [15]:
# ============================================
# 2. PROJECT PATHS
# ============================================

# This notebook should be inside the "notebooks" folder.
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SRC_DIR = PROJECT_ROOT / "src"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "credit_card_fraud.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", DATA_PATH)
print("Processed data folder:", PROCESSED_DIR)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place credit_card_fraud.csv inside data/raw/."
    )


Project root: c:\Users\AIRA\OneDrive\Desktop\241BCADA06\Desktop\ML PROJECT\ML-Final-Lab-Group-05
Raw data path: c:\Users\AIRA\OneDrive\Desktop\241BCADA06\Desktop\ML PROJECT\ML-Final-Lab-Group-05\data\raw\credit_card_fraud.csv
Processed data folder: c:\Users\AIRA\OneDrive\Desktop\241BCADA06\Desktop\ML PROJECT\ML-Final-Lab-Group-05\data\processed


In [16]:
# ============================================
# 3. LOAD RAW DATA
# ============================================

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Shape:", df.shape)

display(df.head())


Dataset loaded successfully.
Rows: 339607
Columns: 15
Shape: (339607, 15)


,trans_date_trans_time,merchant,category,amt,city,state,lat,long,city_pop,job,dob,trans_num,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:44,"Heller, Gutmann and Zieme",grocery_pos,107.23,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,49.159047,-118.186462,0
1,2019-01-01 00:00:51,Lind-Buckridge,entertainment,220.11,Malad City,ID,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,43.150704,-112.154481,0
2,2019-01-01 00:07:27,Kiehn Inc,grocery_pos,96.29,Grenada,CA,41.6125,-122.5258,589,Systems analyst,1945-12-21,413636e759663f264aae1819a4d4f231,41.657520,-122.230347,0
3,2019-01-01 00:09:03,Beier-Hyatt,shopping_pos,7.77,High Rolls Mountain Park,NM,32.9396,-105.8189,899,Naval architect,1967-08-30,8a6293af5ed278dea14448ded2685fea,32.863258,-106.520205,0
4,2019-01-01 00:21:32,Bruen-Yost,misc_pos,6.85,Freedom,WY,43.0172,-111.0292,471,"Education officer, museum",1967-08-02,f3c43d336e92a44fc2fb67058d5949e3,43.753735,-111.454923,0


## 3. Dataset acquisition documentation

The raw dataset is treated as the source dataset. The raw file is kept unchanged in `data/raw/`.

The Data Engineer should not overwrite the raw dataset during cleaning or feature engineering.


In [17]:
# ============================================
# 4. BASIC DATASET INSPECTION
# ============================================

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDataset information:")
df.info()

print("\nDescriptive statistics:")
display(df.describe(include="all").T)


Column names:
['trans_date_trans_time', 'merchant', 'category', 'amt', 'city', 'state', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'merch_lat', 'merch_long', 'is_fraud']

Data types:


,dtype
trans_date_trans_time,str
merchant,str
category,str
amt,float64
city,str
state,str
lat,float64
long,float64
city_pop,int64
job,str



Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 339607 entries, 0 to 339606
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   trans_date_trans_time  339607 non-null  str    
 1   merchant               339607 non-null  str    
 2   category               339607 non-null  str    
 3   amt                    339607 non-null  float64
 4   city                   339607 non-null  str    
 5   state                  339607 non-null  str    
 6   lat                    339607 non-null  float64
 7   long                   339607 non-null  float64
 8   city_pop               339607 non-null  int64  
 9   job                    339607 non-null  str    
 10  dob                    339607 non-null  str    
 11  trans_num              339607 non-null  str    
 12  merch_lat              339607 non-null  float64
 13  merch_long             339607 non-null  float64
 14  is_fraud               33

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
trans_date_trans_time,339607,338504,2019-12-06 20:30:44,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
merchant,339607,693,Kilback LLC,1149,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,339607,14,gas_transport,35089,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amt,339607.0,NaN,NaN,NaN,70.577984,161.675242,1.0,9.6,46.46,83.35,28948.9
city,339607,176,Phoenix,7297,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,339607,13,CA,80495,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,339607.0,NaN,NaN,NaN,39.718991,5.094961,20.0271,36.7154,39.6171,41.71,66.6933
long,339607.0,NaN,NaN,NaN,-110.622605,12.65137,-165.6723,-120.0936,-111.0985,-100.6215,-89.6287
city_pop,339607.0,NaN,NaN,NaN,107140.865515,293029.887292,46.0,471.0,1645.0,35439.0,2383912.0
job,339607,163,"Surveyor, minerals",6589,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# ============================================
# 5. SCHEMA VALIDATION
# ============================================

expected_columns = {
    "trans_date_trans_time",
    "merchant",
    "category",
    "amt",
    "city",
    "state",
    "lat",
    "long",
    "city_pop",
    "job",
    "dob",
    "trans_num",
    "merch_lat",
    "merch_long",
    "is_fraud"
}

actual_columns = set(df.columns)

missing_columns = expected_columns - actual_columns
unexpected_columns = actual_columns - expected_columns

print("Missing columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

assert not missing_columns, f"Missing required columns: {missing_columns}"

print("\nSchema validation passed.")


Missing columns: set()
Unexpected columns: set()

Schema validation passed.


In [19]:
# ============================================
# 6. DATA TYPE CONVERSION
# ============================================

df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"],
    errors="coerce"
)

df["dob"] = pd.to_datetime(
    df["dob"],
    errors="coerce"
)

print("Date columns converted.")
display(df[["trans_date_trans_time", "dob"]].head())

print("\nMissing values created by invalid date conversion:")
display(df[["trans_date_trans_time", "dob"]].isnull().sum())


Date columns converted.


,trans_date_trans_time,dob
0,2019-01-01 00:00:44,1978-06-21
1,2019-01-01 00:00:51,1962-01-19
2,2019-01-01 00:07:27,1945-12-21
3,2019-01-01 00:09:03,1967-08-30
4,2019-01-01 00:21:32,1967-08-02



Missing values created by invalid date conversion:


trans_date_trans_time    0
dob                      0
dtype: int64

In [20]:
# ============================================
# 7. MISSING-VALUE CHECK
# ============================================

missing_counts = df.isnull().sum()

print("Total missing values:", missing_counts.sum())

if missing_counts.sum() == 0:
    print("No missing values found.")
else:
    display(missing_counts[missing_counts > 0])


Total missing values: 0
No missing values found.


### Missing-value handling decision

If the dataset contains no missing values, no artificial imputation is required.

If missing values are discovered, the appropriate treatment must be documented rather than silently deleting rows.


In [21]:
# ============================================
# 8. DUPLICATE CHECKS
# ============================================

duplicate_rows = df.duplicated().sum()
duplicate_transaction_ids = df["trans_num"].duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate transaction IDs:", duplicate_transaction_ids)

if duplicate_rows > 0:
    print("\nDuplicate rows found. Removing exact duplicate rows.")
    df = df.drop_duplicates().reset_index(drop=True)

if duplicate_transaction_ids > 0:
    print("\nWarning: duplicate transaction IDs remain.")
else:
    print("\nTransaction ID uniqueness check passed.")

print("Shape after duplicate handling:", df.shape)


Duplicate rows: 0
Duplicate transaction IDs: 0

Transaction ID uniqueness check passed.
Shape after duplicate handling: (339607, 15)


In [22]:
# ============================================
# 9. INVALID-VALUE VALIDATION
# ============================================

invalid_checks = {
    "negative_amounts": (df["amt"] < 0).sum(),
    "invalid_latitude": ((df["lat"] < -90) | (df["lat"] > 90)).sum(),
    "invalid_longitude": ((df["long"] < -180) | (df["long"] > 180)).sum(),
    "invalid_merchant_latitude": ((df["merch_lat"] < -90) | (df["merch_lat"] > 90)).sum(),
    "invalid_merchant_longitude": ((df["merch_long"] < -180) | (df["merch_long"] > 180)).sum(),
    "negative_city_population": (df["city_pop"] < 0).sum(),
    "invalid_target_values": (~df["is_fraud"].isin([0, 1])).sum()
}

invalid_results = pd.Series(invalid_checks, name="count")

display(invalid_results.to_frame())

assert invalid_checks["invalid_target_values"] == 0,     "Invalid values found in is_fraud."

assert invalid_checks["invalid_latitude"] == 0,     "Invalid customer latitude values found."

assert invalid_checks["invalid_longitude"] == 0,     "Invalid customer longitude values found."

assert invalid_checks["invalid_merchant_latitude"] == 0,     "Invalid merchant latitude values found."

assert invalid_checks["invalid_merchant_longitude"] == 0,     "Invalid merchant longitude values found."

print("Target and geographic validation passed.")


,count
negative_amounts,0
invalid_latitude,0
invalid_longitude,0
invalid_merchant_latitude,0
invalid_merchant_longitude,0
negative_city_population,0
invalid_target_values,0


Target and geographic validation passed.


### Invalid-value treatment

The code validates values that have clear domain constraints.

It does **not** automatically delete unusual transaction amounts because extreme transactions can contain useful fraud information. Outlier detection and fraud-related interpretation should not be confused with invalid-record removal.


In [23]:
# ============================================
# 10. TEXT/CATEGORICAL CLEANING
# ============================================

text_columns = ["merchant", "category", "city", "job"]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

df["category"] = df["category"].str.lower()
df["state"] = df["state"].astype("string").str.strip().str.upper()

print("Text cleaning completed.")

for col in ["category", "state"]:
    print(f"\nSample values in {col}:")
    print(df[col].dropna().unique()[:10])


Text cleaning completed.

Sample values in category:
<StringArray>
[   'grocery_pos',  'entertainment',   'shopping_pos',       'misc_pos',
   'shopping_net',  'gas_transport',       'misc_net',    'grocery_net',
    'food_dining', 'health_fitness']
Length: 10, dtype: string

Sample values in state:
<StringArray>
['WA', 'ID', 'CA', 'NM', 'WY', 'HI', 'NE', 'OR', 'UT', 'AZ']
Length: 10, dtype: string


In [24]:
# ============================================
# 11. TARGET VALIDATION
# ============================================

target_counts = df["is_fraud"].value_counts().sort_index()

print("Target distribution:")
display(target_counts)

print("\nTarget proportions:")
display(df["is_fraud"].value_counts(normalize=True).sort_index())

assert set(df["is_fraud"].unique()).issubset({0, 1})

print("\nTarget validation passed.")


Target distribution:


is_fraud
0    337825
1      1782
Name: count, dtype: int64


Target proportions:


is_fraud
0    0.994753
1    0.005247
Name: proportion, dtype: float64


Target validation passed.


## 12. Feature engineering

The following features are created once and used consistently:

- `age`
- `hour`
- `day_of_week`
- `month`
- `log_amt`
- `distance_km`

The transaction ID is removed because it is an identifier rather than a predictive feature.


In [35]:
# ============================================
# 12. FEATURE ENGINEERING
# ============================================

# Age
df["age"] = (
    (df["trans_date_trans_time"] - df["dob"]).dt.days / 365.25
).round().astype("Int64")

# Time features
df["hour"] = df["trans_date_trans_time"].dt.hour
df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
df["month"] = df["trans_date_trans_time"].dt.month

# Log transaction amount
df["log_amt"] = np.log1p(df["amt"])

# Haversine geographical distance in kilometres
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


df["distance_km"] = haversine_distance(
    df["lat"],
    df["long"],
    df["merch_lat"],
    df["merch_long"]
)

# Remove identifier and DOB after creating age
df = df.drop(columns=["trans_num", "dob"])

print("Feature engineering completed.")
display(df.head())

KeyError: 'dob'

In [36]:
# ============================================
# 13. SAVE FULL CLEANED DATASET
# ============================================

FULL_PROCESSED_PATH = PROCESSED_DIR / "cleaned_full.csv"

df.to_csv(FULL_PROCESSED_PATH, index=False)

print("Full processed dataset saved successfully!")
print("Location:", FULL_PROCESSED_PATH)
print("Shape:", df.shape)

display(df.head())

Full processed dataset saved successfully!
Location: c:\Users\AIRA\OneDrive\Desktop\241BCADA06\Desktop\ML PROJECT\ML-Final-Lab-Group-05\data\processed\cleaned_full.csv
Shape: (339607, 19)


,trans_date_trans_time,merchant,category,amt,city,state,lat,long,city_pop,job,merch_lat,merch_long,is_fraud,age,hour,day_of_week,month,log_amt,distance_km
0,2019-01-01 00:00:44,"Heller, Gutmann and Zieme",grocery_pos,107.23,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,49.159047,-118.186462,0,41,0,1,1,4.684259,30.212176
1,2019-01-01 00:00:51,Lind-Buckridge,entertainment,220.11,Malad City,ID,42.1808,-112.2620,4154,Nature conservation officer,43.150704,-112.154481,0,57,0,1,1,5.398660,108.206083
2,2019-01-01 00:07:27,Kiehn Inc,grocery_pos,96.29,Grenada,CA,41.6125,-122.5258,589,Systems analyst,41.657520,-122.230347,0,73,0,1,1,4.577696,25.059079
3,2019-01-01 00:09:03,Beier-Hyatt,shopping_pos,7.77,High Rolls Mountain Park,NM,32.9396,-105.8189,899,Naval architect,32.863258,-106.520205,0,51,0,1,1,2.171337,66.021685
4,2019-01-01 00:21:32,Bruen-Yost,misc_pos,6.85,Freedom,WY,43.0172,-111.0292,471,"Education officer, museum",43.753735,-111.454923,0,51,0,1,1,2.060514,88.830984


In [37]:
# ============================================
# FINAL DATA QUALITY CHECK
# ============================================

print("Negative values check:")

print("Negative amount:", (df["amt"] < 0).sum())
print("Negative city population:", (df["city_pop"] < 0).sum())
print("Negative age:", (df["age"] < 0).sum())
print("Negative log amount:", (df["log_amt"] < 0).sum())
print("Negative distance:", (df["distance_km"] < 0).sum())

print("\nRange checks:")

print("Latitude:", df["lat"].min(), "to", df["lat"].max())
print("Longitude:", df["long"].min(), "to", df["long"].max())

print("Merchant latitude:", df["merch_lat"].min(), "to", df["merch_lat"].max())
print("Merchant longitude:", df["merch_long"].min(), "to", df["merch_long"].max())

print("Age:", df["age"].min(), "to", df["age"].max())
print("Hour:", df["hour"].min(), "to", df["hour"].max())
print("Day of week:", df["day_of_week"].min(), "to", df["day_of_week"].max())
print("Month:", df["month"].min(), "to", df["month"].max())

print("\nMissing values:")
print(df.isna().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Negative values check:
Negative amount: 0
Negative city population: 0
Negative age: 0
Negative log amount: 0
Negative distance: 0

Range checks:
Latitude: 20.0271 to 66.6933
Longitude: -165.6723 to -89.6287
Merchant latitude: 19.027422 to 67.510267
Merchant longitude: -166.671575 to -88.629203
Age: 17 to 93
Hour: 0 to 23
Day of week: 0 to 6
Month: 1 to 12

Missing values:
0

Duplicate rows:
0


## 13. Outlier strategy

The raw transaction amount is strongly right-skewed, so `log_amt` is created.

We do **not** blindly delete high-value transactions because an extreme transaction can be a genuine transaction or a useful fraud signal.

Most importantly, preprocessing parameters that are learned from the data must not be calculated using the test set.


In [ ]:
# ============================================
# 13. PREPARE FEATURES AND TARGET
# ============================================

target = "is_fraud"

X = df.drop(columns=[target])
y = df[target].astype(int)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

assert target not in X.columns
assert len(X) == len(y)

print("Target successfully separated from features.")


Feature shape: (339607, 18)
Target shape: (339607,)
Target successfully separated from features.


In [ ]:
# ============================================
# 14. TRAIN/TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nFraud percentage:")
print("Train:", round(y_train.mean() * 100, 4), "%")
print("Test :", round(y_test.mean() * 100, 4), "%")

print("\nStratification check:")
print("Difference:", abs(y_train.mean() - y_test.mean()))


X_train: (271685, 18)
X_test : (67922, 18)
y_train: (271685,)
y_test : (67922,)

Fraud percentage:
Train: 0.5249 %
Test : 0.5241 %

Stratification check:
Difference: 7.419341953411596e-06


### Why the split happens before learned preprocessing

The train/test split is performed before fitting the scaler or encoder.

This prevents information from the test set from being used to learn preprocessing parameters.

This is an important **data-leakage prevention** step.


In [ ]:
# ============================================
# 15. DEFINE PREPROCESSING COLUMNS
# ============================================

# Columns that should be treated as numerical
numeric_features = [
    "amt",
    "log_amt",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "age",
    "hour",
    "day_of_week",
    "month",
    "distance_km"
]

# Lower-cardinality categorical feature retained for preprocessing
categorical_features = [
    "category"
]

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

missing_numeric = set(numeric_features) - set(X_train.columns)
missing_categorical = set(categorical_features) - set(X_train.columns)

assert not missing_numeric, f"Missing numerical features: {missing_numeric}"
assert not missing_categorical, f"Missing categorical features: {missing_categorical}"


Numerical features:
['amt', 'log_amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'age', 'hour', 'day_of_week', 'month', 'distance_km']

Categorical features:
['category']


In [ ]:
# ============================================
# 16. LEAKAGE-SAFE PREPROCESSING PIPELINE
# ============================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    remainder="drop"
)

# IMPORTANT:
# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the already-fitted preprocessing
X_test_processed = preprocessor.transform(X_test)

print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape :", X_test_processed.shape)
print("Leakage-safe preprocessing completed.")


Training processed shape: (271685, 25)
Testing processed shape : (67922, 25)
Leakage-safe preprocessing completed.


In [ ]:
# ============================================
# 17. CREATE PROCESSED FEATURE NAMES
# ============================================

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Processed feature count:", len(feature_names))

display(X_train_processed.head())


Processed feature count: 25


,num__amt,num__log_amt,num__lat,num__long,num__city_pop,num__merch_lat,num__merch_long,num__age,num__hour,num__day_of_week,num__month,num__distance_km,cat__category_food_dining,cat__category_gas_transport,cat__category_grocery_net,cat__category_grocery_pos,cat__category_health_fitness,cat__category_home,cat__category_kids_pets,cat__category_misc_net,cat__category_misc_pos,cat__category_personal_care,cat__category_shopping_net,cat__category_shopping_pos,cat__category_travel
179685,-0.289910,-0.214683,0.376172,1.112758,-0.360142,0.200292,1.064698,-0.106740,0.027640,0.930757,-1.505925,1.210972,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
138301,-0.368123,-0.720473,0.376172,1.112758,-0.360142,0.409618,1.068315,-0.106740,1.201871,1.387281,1.121452,-0.884341,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
200559,0.237508,0.901195,-0.393077,-0.146548,-0.365206,-0.368350,-0.131477,0.597831,-0.412697,-1.351863,-1.213994,-1.883408,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39697,0.687091,1.288689,-1.177480,-0.451254,-0.350158,-1.338050,-0.467633,0.950117,-0.999812,0.930757,-0.922063,0.786061,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
84427,-0.388513,-0.933542,0.153782,-1.053987,-0.362206,0.150038,-1.091945,1.243688,-1.146591,-1.351863,-0.046271,-1.158758,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [ ]:
# ============================================
# 18. FINAL PROCESSED DATA VALIDATION
# ============================================

print("X_train shape:", X_train_processed.shape)
print("X_test shape :", X_test_processed.shape)

print("\nMissing values in X_train:",
      X_train_processed.isnull().sum().sum())

print("Missing values in X_test:",
      X_test_processed.isnull().sum().sum())

print("\nTarget values:")
print("y_train:", y_train.value_counts().to_dict())
print("y_test :", y_test.value_counts().to_dict())

assert X_train_processed.isnull().sum().sum() == 0
assert X_test_processed.isnull().sum().sum() == 0

assert len(X_train_processed) == len(y_train)
assert len(X_test_processed) == len(y_test)

print("\nFinal processed-data validation passed.")


X_train shape: (271685, 25)
X_test shape : (67922, 25)

Missing values in X_train: 0
Missing values in X_test: 0

Target values:
y_train: {0: 270259, 1: 1426}
y_test : {0: 67566, 1: 356}

Final processed-data validation passed.


In [32]:
# ============================================
# 19. SAVE TRAIN/TEST DATA
# ============================================

train_final = X_train_processed.copy()
train_final["is_fraud"] = y_train.values

test_final = X_test_processed.copy()
test_final["is_fraud"] = y_test.values

X_train_path = PROCESSED_DIR / "X_train.csv"
X_test_path = PROCESSED_DIR / "X_test.csv"
train_path = PROCESSED_DIR / "train_processed.csv"
test_path = PROCESSED_DIR / "test_processed.csv"

X_train_processed.to_csv(X_train_path, index=False)
X_test_processed.to_csv(X_test_path, index=False)

train_final.to_csv(train_path, index=False)
test_final.to_csv(test_path, index=False)

print("Saved files:")
print(X_train_path)
print(X_test_path)
print(train_path)
print(test_path)


Saved files:
c:\Users\AIRA\OneDrive\Desktop\241BCADA06\Desktop\ML PROJECT\ML-Final-Lab-Group-05\data\processed\X_train.csv
c:\Users\AIRA\OneDrive\Desktop\241BCADA06\Desktop\ML PROJECT\ML-Final-Lab-Group-05\data\processed\X_test.csv
c:\Users\AIRA\OneDrive\Desktop\241BCADA06\Desktop\ML PROJECT\ML-Final-Lab-Group-05\data\processed\train_processed.csv
c:\Users\AIRA\OneDrive\Desktop\241BCADA06\Desktop\ML PROJECT\ML-Final-Lab-Group-05\data\processed\test_processed.csv


In [33]:
# ============================================
# 20. DATA-QUALITY AUDIT LOG
# ============================================

quality_log = pd.DataFrame({
    "Check": [
        "Expected schema",
        "Missing values",
        "Duplicate rows before cleaning",
        "Duplicate transaction IDs",
        "Negative transaction amounts",
        "Invalid customer latitude",
        "Invalid customer longitude",
        "Invalid merchant latitude",
        "Invalid merchant longitude",
        "Negative city population",
        "Invalid target values"
    ],
    "Result": [
        "Passed" if not missing_columns else str(missing_columns),
        int(missing_counts.sum()),
        int(duplicate_rows),
        int(duplicate_transaction_ids),
        int(invalid_checks["negative_amounts"]),
        int(invalid_checks["invalid_latitude"]),
        int(invalid_checks["invalid_longitude"]),
        int(invalid_checks["invalid_merchant_latitude"]),
        int(invalid_checks["invalid_merchant_longitude"]),
        int(invalid_checks["negative_city_population"]),
        int(invalid_checks["invalid_target_values"])
    ]
})

display(quality_log)

quality_log_path = PROCESSED_DIR / "data_quality_log.csv"
quality_log.to_csv(quality_log_path, index=False)

print("Audit log saved to:", quality_log_path)


,Check,Result
0,Expected schema,Passed
1,Missing values,0
2,Duplicate rows before cleaning,0
3,Duplicate transaction IDs,0
4,Negative transaction amounts,0
5,Invalid customer latitude,0
6,Invalid customer longitude,0
7,Invalid merchant latitude,0
8,Invalid merchant longitude,0
9,Negative city population,0


Audit log saved to: c:\Users\AIRA\OneDrive\Desktop\241BCADA06\Desktop\ML PROJECT\ML-Final-Lab-Group-05\data\processed\data_quality_log.csv


## 21. Reproducibility check

This section verifies that the final train/test datasets have:

- matching feature columns
- no missing processed values
- separated target
- reproducible random state
- consistent preprocessing
- no use of the test set when fitting the preprocessing transformer


In [34]:
# ============================================
# 21. REPRODUCIBILITY / FINAL CHECKS
# ============================================

assert list(X_train_processed.columns) == list(X_test_processed.columns)

assert "is_fraud" not in X_train_processed.columns
assert "is_fraud" not in X_test_processed.columns

assert X_train_processed.shape[1] == X_test_processed.shape[1]

assert len(X_train_processed) == len(y_train)
assert len(X_test_processed) == len(y_test)

print("============================================")
print("MEMBER 1 DATA ENGINEERING CHECKS PASSED")
print("============================================")
print("Schema validation       : PASS")
print("Missing-value handling  : PASS")
print("Duplicate handling      : PASS")
print("Invalid-value checks    : PASS")
print("Feature engineering     : PASS")
print("Train/test split        : PASS")
print("Stratification          : PASS")
print("Leakage-safe preprocessing: PASS")
print("Processed data saved    : PASS")
print("Quality log saved       : PASS")


MEMBER 1 DATA ENGINEERING CHECKS PASSED
Schema validation       : PASS
Missing-value handling  : PASS
Duplicate handling      : PASS
Invalid-value checks    : PASS
Feature engineering     : PASS
Train/test split        : PASS
Stratification          : PASS
Leakage-safe preprocessing: PASS
Processed data saved    : PASS
Quality log saved       : PASS
